# 05. 에이전트 (Agent) 입문

지금까지 쓴 LLM 은 질문을 받으면 **글을 써서 돌려주는 것** 밖에 못 합니다. 계산도, 표 조회도, 논문 검색도 "머릿속으로" 흉내 낼 뿐입니다.
**에이전트(Agent)** 는 LLM 에게 **도구(Tool)** 를 쥐여 주고, 필요할 때 스스로 도구를 골라 쓰게 만든 것입니다.

| | LLM | 에이전트 |
|---|---|---|
| 비유 | 실험실에 앉아 **말만 하는 두뇌** | 계산기·물성표·문헌검색을 **직접 쓰는 연구원** |
| 계산 | 암산(자주 틀림) | 계산기 도구 사용 |
| 물성값 | 기억에 의존(지어낼 수 있음) | 물성표 도구에서 조회 |
| 문헌 | 학습한 기억만 | 논문 검색 도구 사용 |

## 학습 목표

1. **도구(Tool)** = 파이썬 함수 + 설명서(docstring) 라는 것을 이해하고 `@tool` 로 도구 3개를 만듭니다.
2. `bind_tools` 로 LLM 이 **"어떤 도구를 어떤 인자로 부를지"** 결정하는 모습(`tool_calls`)을 확인합니다.
3. **ReAct 루프**(생각 → 행동 → 관찰 반복)를 `while` 문으로 **직접** 구현합니다.
4. 같은 것을 LangChain 의 `create_agent` 로 만들고, 둘이 같은 일을 한다는 것을 확인합니다.
5. 에이전트가 **틀리는 경우** 와 그 이유를 이해합니다.

## 환경설정

In [ ]:
# [Colab 전용] 저장소를 내려받고 패키지·Ollama·모델을 설치합니다. (로컬 Jupyter에서는 자동으로 건너뜁니다)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/skku_rag_tutorials"):
        !git clone -q https://github.com/JDeun/skku_rag_tutorials.git /content/skku_rag_tutorials
    %cd /content/skku_rag_tutorials
    !bash scripts/colab_setup.sh

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # 불필요한 경고 숨기기

from skku_rag import get_llm, LLM_MODEL

llm = get_llm()  # Ollama 로컬 언어모델
print("LLM 모델:", LLM_MODEL)

## 1. LLM 은 '말만 하는 두뇌'

배선 저항 계산 문제를 LLM 에게 그냥 물어봅시다. 정답을 알려면 ① Ru 의 비저항 값을 **정확히** 알고 ② 단위를 맞춰 **계산** 해야 합니다.

> 정답(리뷰 논문 Table 1 의 Ru 비저항 7.8 μΩ·cm 사용): R = ρL/A = (7.8×10⁻⁸ Ω·m × 10×10⁻⁶ m) / (10×10⁻⁹ m × 20×10⁻⁹ m) = **3,900 Ω**

In [ ]:
question = "Ru(루테늄)로 길이 10 µm, 단면 10 nm × 20 nm 인 배선을 만들면 저항은 몇 옴(Ω)인가요? 간단히 답하세요."
print(llm.invoke(question).content)

소형 LLM 은 비저항 값을 기억에서 꺼내고(틀릴 수 있음), 10의 거듭제곱 계산을 "글로" 합니다. 맞을 때도 있지만, **맞았는지 확인할 방법이 없다** 는 것이 문제입니다.
연구에서 이런 숫자를 그대로 쓸 수는 없겠죠. 그래서 도구가 필요합니다.

## 2. 도구(Tool)란 — 파이썬 함수 + 설명서

도구는 특별한 것이 아닙니다. **평범한 파이썬 함수** 에 **설명서(docstring)** 를 붙인 것입니다.

```
@tool                                   ← "이 함수를 도구로 등록" 표시
def 함수이름(인자: 타입) -> str:         ← 이름·인자·타입이 LLM 에게 그대로 보임
    """이 도구가 하는 일, 인자의 뜻·단위"""   ← LLM 이 '언제 이 도구를 쓸지' 판단하는 유일한 근거
    ...                                  ← 실제 일은 파이썬이 함(LLM 아님)
```

장비 옆에 붙어 있는 **사용 설명서** 를 떠올리세요. 설명서가 엉망이면 신입 연구원(LLM)은 장비를 잘못 씁니다.

### 도구 ① 금속 물성 조회

리뷰 논문(Sung et al., *J. Mater. Chem. C* 2026) **Table 1** 의 값을 작은 사전(dict)으로 옮겨 둡니다. 실제 연구라면 02장의 데이터베이스(RDB)를 조회하겠지만, 여기서는 간단히 하드코딩합니다.

In [ ]:
# 출처: Sung et al., "Recent progress in alternative metals for advanced interconnects",
#       J. Mater. Chem. C 2026, Table 1 (상온 벌크 비저항 ρ0, 전자 평균자유행로 EMFP λ)
#       hcp 금속(Co, Ru)은 방향에 따라 EMFP 가 두 값(a/c 축)이라 큰 값을 적었습니다.
METAL_TABLE = {
    "Cu": {"structure": "fcc", "resistivity_uohm_cm": 1.678, "emfp_nm": 39.9},
    "Al": {"structure": "fcc", "resistivity_uohm_cm": 2.650, "emfp_nm": 18.9},
    "Rh": {"structure": "fcc", "resistivity_uohm_cm": 4.7, "emfp_nm": 6.88},
    "Ir": {"structure": "fcc", "resistivity_uohm_cm": 5.2, "emfp_nm": 7.09},
    "Ni": {"structure": "fcc", "resistivity_uohm_cm": 6.93, "emfp_nm": 5.87},
    "Co": {"structure": "hcp", "resistivity_uohm_cm": 6.2, "emfp_nm": 11.8},
    "Ru": {"structure": "hcp", "resistivity_uohm_cm": 7.8, "emfp_nm": 6.59},
    "Mo": {"structure": "bcc", "resistivity_uohm_cm": 5.34, "emfp_nm": 11.2},
}

In [ ]:
from langchain_core.tools import tool

@tool
def get_metal_property(metal: str) -> str:
    """금속의 결정구조, 상온 벌크 비저항(μΩ·cm), 전자 평균자유행로 EMFP(nm)를 물성표에서 조회합니다.
    metal 에는 원소기호를 넣으세요. 예: "Cu", "Ru", "Co", "Mo"."""
    row = METAL_TABLE.get(metal.strip().capitalize())
    if row is None:  # 모르는 입력이면, 무엇을 넣어야 하는지 알려 주는 오류 메시지를 돌려줌
        return f"'{metal}' 은(는) 물성표에 없습니다. 사용 가능한 원소기호: {', '.join(METAL_TABLE)}"
    return f"{metal}: 결정구조 {row['structure']}, 비저항 {row['resistivity_uohm_cm']} μΩ·cm, EMFP {row['emfp_nm']} nm"

### 도구 ② 배선 저항 계산 (R = ρL/A)

단위 변환은 사람도 자주 실수하는 부분이라 **도구 안에서** 처리합니다. 인자 이름에 단위를 넣어(`_um`, `_nm`) LLM 이 헷갈리지 않게 합니다.

In [ ]:
@tool
def calc_wire_resistance(resistivity_uohm_cm: float, length_um: float, width_nm: float, height_nm: float) -> str:
    """직사각형 단면 배선의 전기저항 R = ρL/A 를 계산합니다(단위 변환 포함).
    resistivity_uohm_cm: 비저항(μΩ·cm), length_um: 길이(µm), width_nm·height_nm: 단면 가로·세로(nm)."""
    rho = resistivity_uohm_cm * 1e-8           # μΩ·cm → Ω·m  (1 μΩ·cm = 1e-8 Ω·m)
    length = length_um * 1e-6                  # µm → m
    area = (width_nm * 1e-9) * (height_nm * 1e-9)  # nm × nm → m²
    resistance = rho * length / area
    return f"R = {resistance:,.1f} Ω  (ρ={resistivity_uohm_cm} μΩ·cm, L={length_um} µm, A={width_nm}×{height_nm} nm²)"

### 도구 ③ 논문 키워드 검색

`load_papers()` 로 읽은 논문 페이지에서 **단순 문자열 검색**(Ctrl+F)을 합니다. 06장의 벡터 검색보다 훨씬 원시적이지만, 도구가 무엇이든 될 수 있다는 것을 보여 줍니다.

In [ ]:
from pathlib import Path
from skku_rag import load_papers

PAGES = load_papers()  # 논문 페이지 목록(한 번만 읽어 둠)

@tool
def search_papers(keyword: str) -> str:
    """논문 2편(나노결정 Cu 합금 논문, 대체 배선 금속 리뷰)에서 영어 키워드를 Ctrl+F 처럼 찾아
    해당 부분 앞뒤 문장을 최대 3개 돌려줍니다. 예: "electromigration", "grain boundary"."""
    hits = []
    for page in PAGES:
        text = " ".join(page.page_content.split())   # 줄바꿈을 공백으로 정리
        pos = text.lower().find(keyword.lower())
        if pos >= 0:
            name = Path(page.metadata["source"]).name[:30]
            hits.append(f"[{name}, p.{page.metadata['page'] + 1}] ...{text[max(0, pos - 150): pos + 250]}...")
    return "\n".join(hits[:3]) if hits else f"'{keyword}' 를 논문에서 찾지 못했습니다. 다른 영어 키워드로 시도하세요."

### 도구 들여다보기 & 직접 실행해 보기

`@tool` 을 붙이면 함수가 **이름(name)·설명(description)·인자 형식(args)** 을 가진 도구 객체가 됩니다. LLM 에게 전달되는 것은 바로 이 정보입니다.
도구는 LLM 없이도 `.invoke({인자})` 로 직접 실행해 볼 수 있습니다.

In [ ]:
tools = [get_metal_property, calc_wire_resistance, search_papers]

for t in tools:
    print(f"■ {t.name}  인자: {list(t.args)}")
    print("  ", t.description.splitlines()[0])

In [ ]:
print(get_metal_property.invoke({"metal": "Ru"}))
print(get_metal_property.invoke({"metal": "루테늄"}))   # 한글 이름 → 친절한 오류 메시지
print(calc_wire_resistance.invoke({"resistivity_uohm_cm": 7.8, "length_um": 10, "width_nm": 10, "height_nm": 20}))
print(search_papers.invoke({"keyword": "electromigration"})[:300])

계산 도구는 정답 **3,900 Ω** 을 정확히 냅니다. 이제 LLM 이 이 도구들을 **스스로 골라 쓰게** 만들면 됩니다.

## 3. `bind_tools` — LLM 이 도구를 '주문' 하는 모습

`llm.bind_tools(tools)` 는 LLM 에게 "이런 도구들이 있다" 는 **도구 목록(설명서 묶음)** 을 함께 보내도록 설정합니다.
중요한 점: **LLM 은 도구를 직접 실행하지 않습니다.** "이 도구를 이 인자로 실행해 주세요" 라는 **주문서(`tool_calls`)** 만 써서 돌려줍니다.
실행은 우리 파이썬 코드가 합니다. 연구원(LLM)이 실험 의뢰서를 쓰면, 장비 담당자(파이썬)가 측정하는 구조입니다.

In [ ]:
llm_with_tools = llm.bind_tools(tools)  # 도구 목록을 LLM 에 연결

ai_msg = llm_with_tools.invoke("코발트(Co)의 전자 평균자유행로는 얼마인가요?")
print("글 답변(content):", repr(ai_msg.content))
print("도구 주문서(tool_calls):", ai_msg.tool_calls)

- 대개 `content` 는 비어 있고(또는 짧고), `tool_calls` 에 **도구 이름** 과 **인자** 가 들어 있습니다. 여러분의 결과에서 어떤 도구를 골랐는지 확인해 보세요.
- 한국어 "코발트" 를 보고 원소기호 `"Co"` 로 바꿔 넣었다면, 그건 docstring 에 "원소기호를 넣으세요" 라고 써 둔 덕분입니다.

도구가 필요 없는 질문에는 보통 주문서 없이 그냥 답합니다(작은 모델은 가끔 불필요한 도구를 주문하기도 합니다).

In [ ]:
ai_msg2 = llm_with_tools.invoke("안녕하세요! 한 문장으로 자기소개 해 주세요.")
print("content   :", ai_msg2.content)
print("tool_calls:", ai_msg2.tool_calls)

## 4. ReAct 루프 직접 구현하기

주문서를 받았으면 ① 도구를 실행하고 ② 결과를 LLM 에게 다시 보여 주고 ③ LLM 이 다음 행동을 정하게 합니다. 이것을 **답이 나올 때까지 반복** 하는 것이 **ReAct(Reasoning + Acting)** 패턴입니다.

```
            ┌──────────────────────────────────────────────┐
            ▼                                              │
 질문 ─► [생각 Thought] LLM: "어떤 도구가 필요하지?"         │
            │                                              │
            ├─ tool_calls 있음 ─► [행동 Action] 파이썬이 도구 실행
            │                         │                    │
            │                         ▼                    │
            │                    [관찰 Observation] 결과를 메시지에 추가 ─┘
            │
            └─ tool_calls 없음 ─► 최종 답변 (루프 종료)
```

실험에 비유하면 "가설 → 측정 → 결과 확인 → 다음 측정 결정" 의 반복입니다. 무한 반복을 막기 위해 **최대 반복 횟수** 를 둡니다.

대화 기록은 **메시지 목록** 으로 관리합니다.

| 메시지 종류 | 누가 | 내용 |
|---|---|---|
| `SystemMessage` | 우리 | 역할·규칙 지시 |
| `HumanMessage` | 사용자 | 질문 |
| `AIMessage` | LLM | 답변 또는 도구 주문서(`tool_calls`) |
| `ToolMessage` | 도구 | 도구 실행 결과(관찰) |

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

SYSTEM_PROMPT = (
    "당신은 재료공학 연구실의 조교입니다. 물성값은 반드시 get_metal_property 로 조회하고, "
    "계산은 반드시 calc_wire_resistance 로 하세요. 암산하거나 값을 지어내지 마세요. 최종 답은 한국어로 짧게."
)
TOOLS_BY_NAME = {t.name: t for t in tools}  # 이름 → 도구 객체 (주문서의 이름으로 도구를 찾기 위함)

def run_tool(call: dict) -> str:
    """주문서 한 장을 실행. 없는 도구·잘못된 인자여도 프로그램이 멈추지 않게 오류를 글로 돌려줌."""
    tool_obj = TOOLS_BY_NAME.get(call["name"])
    if tool_obj is None:
        return f"오류: '{call['name']}' 라는 도구는 없습니다. 사용 가능: {list(TOOLS_BY_NAME)}"
    try:
        return str(tool_obj.invoke(call["args"]))
    except Exception as e:
        return f"도구 실행 오류: {e}"

In [ ]:
def react_agent(question: str, max_steps: int = 6) -> list:
    """ReAct 루프를 직접 실행하고, 전체 메시지 기록을 돌려줍니다."""
    messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(question)]
    step = 0
    while step < max_steps:                                  # 최대 반복 횟수까지
        step += 1
        ai = llm_with_tools.invoke(messages)                  # [생각] LLM 이 다음 행동 결정
        messages.append(ai)
        if not ai.tool_calls:                                 # 주문서가 없으면 = 최종 답
            print(f"[{step}단계 · 최종 답변] {ai.content}")
            return messages
        for call in ai.tool_calls:                            # [행동] 주문서마다 도구 실행
            print(f"[{step}단계 · 행동] {call['name']}({call['args']})")
            result = run_tool(call)
            print(f"[{step}단계 · 관찰] {result[:200]}")
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))  # [관찰] 결과를 기록에 추가
    print(f"최대 반복 횟수({max_steps})에 도달해 멈췄습니다.")
    return messages

In [ ]:
history = react_agent(question)

에이전트가 ① 물성표에서 Ru 의 비저항을 조회하고 ② 그 값으로 계산 도구를 불러 ③ 결과를 보고 답하는 흐름이 보이나요?
(모델이 두 도구를 한 단계에서 동시에 주문하기도 하고, 단계별로 하나씩 주문하기도 합니다. 동시에 주문할 때는 비저항을 "기억" 으로 넣었을 수 있으니 관찰 결과의 ρ 값을 확인해 보세요.)

전체 **메시지 기록** 을 출력해 보면, 에이전트의 "생각의 흐름" 이 모두 이 목록 안에 쌓인다는 것을 알 수 있습니다. LLM 은 매 단계마다 **이 기록 전체** 를 다시 읽고 다음 행동을 정합니다.

In [ ]:
for i, m in enumerate(history):
    kind = type(m).__name__
    body = m.content if m.content else f"(도구 주문: {[(c['name'], c['args']) for c in m.tool_calls]})"
    print(f"{i}. {kind:14s} | {str(body)[:110]}")

> **참고 — '생각(Thought)' 이 안 보이는 이유**: 원래 ReAct 에서는 LLM 이 "먼저 비저항을 찾아야겠다" 같은 생각을 글로 쓰고 행동합니다.
> 이 튜토리얼의 `get_llm()` 은 속도를 위해 생각 과정 출력(reasoning)을 꺼 두었기 때문에, 생각은 모델 내부에서만 일어나고 우리에게는 **행동(도구 주문)** 만 보입니다.

## 5. 같은 것을 `create_agent` 로

LangChain 의 `create_agent` 는 방금 만든 `while` 루프(LLM 호출 → 도구 실행 → 결과 추가 → 반복)를 미리 만들어 둔 것입니다.
내부적으로는 LangGraph 의 **그래프**(노드: `model`, `tools`)로 구현되어 있습니다.

```
 START ─► [model] ──tool_calls 있음──► [tools] ─┐
            ▲  │                                │
            │  └──tool_calls 없음──► END        │
            └───────────────────────────────────┘
```

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

`stream(..., stream_mode="updates")` 를 쓰면 그래프의 **노드가 하나 끝날 때마다** 결과를 받아볼 수 있습니다. 우리 `while` 루프의 출력과 비교해 보세요.

In [ ]:
def show_stream(agent, question: str) -> None:
    """에이전트 실행 과정을 노드(model/tools) 단위로 출력"""
    inputs = {"messages": [{"role": "user", "content": question}]}
    for update in agent.stream(inputs, stream_mode="updates"):
        for node, data in update.items():
            for m in data["messages"]:
                if getattr(m, "tool_calls", None):
                    print(f"[{node}] 도구 주문:", [(c["name"], c["args"]) for c in m.tool_calls])
                else:
                    print(f"[{node}] {str(m.content)[:200]}")

show_stream(agent, question)

`[model]` → `[tools]` → `[model]` ... 순서가 우리가 만든 루프의 [생각] → [행동/관찰] → [생각] 과 정확히 대응합니다.
**라이브러리는 마법이 아니라, 방금 만든 while 루프를 깔끔하게 포장한 것** 입니다.

## 6. 여러 도구가 필요한 질문

두 금속을 비교하려면 물성 조회 2번 + 계산 2번이 필요합니다. 에이전트가 필요한 만큼 도구를 반복해서 쓰는지 봅시다.

In [ ]:
compare_q = ("Cu 와 Ru 로 각각 길이 10 µm, 단면 10 nm × 20 nm 배선을 만들면 저항은 각각 몇 Ω 인가요? "
             "어느 쪽이 더 작은지도 알려 주세요.")
show_stream(agent, compare_q)

정답: Cu 약 **839 Ω**, Ru 약 **3,900 Ω** (벌크 비저항 기준) → Cu 가 더 작습니다.

> **재료공학적으로 한 번 더 생각하기**: 위 계산은 **벌크 비저항** 을 썼습니다. 그런데 리뷰 논문의 핵심 주장은 선폭이 EMFP(Cu 약 40 nm) 보다 작아지면
> 표면·결정립계 산란 때문에 Cu 의 비저항이 크게 올라가고, EMFP 가 짧은 Ru 는 덜 올라간다는 것입니다.
> 즉 **도구가 정확해도, 도구에 넣은 모델(가정)이 틀리면 답도 틀립니다.** 도구 설계자가 재료 지식을 갖춰야 하는 이유입니다.

에이전트에게 문헌 검색 도구까지 쓰게 해 봅시다.

In [ ]:
show_stream(agent, "논문에서 'size effect' 를 검색해서, 배선이 가늘어질 때 구리 비저항이 어떻게 되는지 2문장으로 설명해 주세요.")

## 7. 에이전트가 틀리는 경우와 이유

에이전트는 만능이 아닙니다. 특히 4B 같은 **소형 모델** 에서 흔한 실패 유형은 다음과 같습니다.

| 실패 유형 | 예 | 주된 원인 |
|---|---|---|
| 도구를 안 씀 | 계산을 암산으로 해 버림 | 지시(시스템 프롬프트)가 약하거나 모델이 작음 |
| 엉뚱한 도구·인자 | `metal="루테늄"`, 단위 혼동(Ω·m 값을 μΩ·cm 자리에) | **도구 설명서(docstring)가 모호함** |
| 도구 결과를 무시 | 조회한 값과 다른 숫자로 답함 | 소형 모델의 긴 맥락 처리 한계 |
| 무한 반복 | 같은 검색을 계속 반복 | 종료 조건 판단 실패 → **최대 반복 횟수** 로 방지 |

### 실험: 설명서가 엉망인 도구

`get_metal_property` 와 **똑같은 일** 을 하지만, 이름과 설명이 엉망인 도구를 만들어 봅시다.

In [ ]:
@tool
def func1(x: str) -> str:
    """데이터를 반환합니다."""
    return get_metal_property.invoke({"metal": x})

bad_llm = llm.bind_tools([func1, calc_wire_resistance])
good_llm = llm.bind_tools([get_metal_property, calc_wire_resistance])

q = "몰리브덴의 상온 비저항은 얼마인가요?"
print("설명서 엉망:", bad_llm.invoke(q).tool_calls or "(도구 안 씀)")
print("설명서 좋음:", good_llm.invoke(q).tool_calls or "(도구 안 씀)")

결과는 실행할 때마다 조금씩 다를 수 있지만, 일반적으로 설명서가 엉망이면
- 이 도구가 비저항을 알려 준다는 것을 몰라 **도구를 쓰지 않거나**,
- 인자 `x` 에 무엇을 넣을지 몰라 `"몰리브덴"` 처럼 **물성표가 모르는 값** 을 넣습니다.

LLM 이 도구에 대해 아는 것은 **이름·docstring·인자 이름과 타입이 전부** 입니다. 사람에게 장비 사용법을 가르치듯, 도구 설명서를 구체적으로(무엇을, 어떤 단위로, 예시까지) 쓰는 것이 에이전트 성능의 절반입니다.

또 하나의 요령은 `get_metal_property` 처럼 **실패했을 때 "무엇을 넣어야 하는지" 알려 주는 오류 메시지** 를 돌려주는 것입니다. 그러면 에이전트가 관찰 결과를 보고 스스로 고쳐서 다시 시도할 수 있습니다.

## 8. 다음 단계 예고 — Agentic RAG (08장)

이번 장의 `search_papers` 는 Ctrl+F 수준의 단순 검색이었습니다. 영어 키워드를 정확히 넣어야만 찾고, 의미가 비슷한 표현은 못 찾습니다.

**08 Agentic RAG** 에서는 06장에서 만든 **벡터 검색기(Retriever) 자체를 도구로** 에이전트에게 줍니다.

```
06 벡터 RAG   : 질문 ─► 검색 1번 ─► 답변                       (항상 정해진 순서)
08 Agentic RAG: 질문 ─► [에이전트] ─► 검색? 다시 검색? 물성표? 계산? ─► 답변  (에이전트가 판단)
```

그러면 06장 C-4 에서 벡터 RAG 가 어려워하는 "여러 조각에 흩어진 정보 모으기·수치 비교" 문제를, 에이전트가 **여러 번 검색하고 도구로 계산** 해서 풀 수 있게 됩니다.

## 정리

- **도구(Tool) = 파이썬 함수 + 설명서(docstring)**. `@tool` 을 붙이면 이름·설명·인자 형식이 LLM 에게 전달됩니다.
- `bind_tools` 를 하면 LLM 은 도구를 **직접 실행하지 않고** `tool_calls`(주문서)만 씁니다. 실행은 파이썬이 합니다.
- **ReAct 루프** = [생각: LLM 호출] → [행동: 도구 실행] → [관찰: ToolMessage 추가] 를 `tool_calls` 가 없을 때까지 반복. 최대 반복 횟수로 안전장치.
- `create_agent` 는 이 while 루프를 LangGraph 그래프(`model` ↔ `tools`)로 포장한 것입니다.
- 소형 모델은 도구를 안 쓰거나 인자를 틀리게 넣을 수 있습니다. **좋은 도구 설명서, 단위가 드러나는 인자 이름, 친절한 오류 메시지** 가 이를 크게 줄여 줍니다.
- 도구가 정확해도 그 안의 물리 모델(예: 벌크 비저항 가정)이 틀리면 답이 틀립니다. 도메인 지식이 여전히 중요합니다.

## 직접 해보기

1. **도구 추가** : 두 금속의 `비저항 × EMFP`(ρ0·λ, 리뷰 논문의 성능 지표)를 계산하는 도구 `calc_rho_lambda(metal: str)` 를 만들어 `tools` 에 추가하고, "Cu 와 Ru 중 ρ0·λ 가 작은 금속은?" 을 `react_agent` 로 물어보세요. (Table 1 정답: Cu 6.70, Ru 5.14 ×10⁻¹⁶ Ω·m²)
2. **설명서 실험** : `calc_wire_resistance` 의 docstring 에서 단위 설명을 지우고 인자 이름을 `a, b, c, d` 로 바꾼 뒤 같은 질문을 해 보세요. 답이 어떻게 달라지나요?
3. **반복 횟수 실험** : `react_agent(compare_q, max_steps=2)` 처럼 최대 반복 횟수를 줄이면 어떤 일이 생기는지 확인하고, 왜 이런 안전장치가 필요한지 설명해 보세요.